# bucket distributions

In [1]:

import duckdb
from pathlib import Path

POS_PATH = Path(
    "../../../datasets/balduf_anon_march_2026/"
    "positive_blocks_analysis_10d_mar2026_v3_anon.parquet"
)

SAMPLE_SIZE = 1000

con = duckdb.connect()

print("\n=== POSITIVE BUCKET SIZE DISTRIBUTION ===")
print(f"Source: {POS_PATH}")
print(f"Target sample size: {SAMPLE_SIZE}")

# TOTAL POSITIVES

total = con.execute(f"""
    SELECT COUNT(*) AS n_pos
    FROM read_parquet('{POS_PATH}')
    WHERE target = 1
""").fetchone()[0]

# BUCKET DISTRIBUTION

bucket_dist = con.execute(f"""
    WITH bucket_counts AS (
        SELECT
            exposure_bucket_index,
            post_bucket_index,
            follow_10d_bucket_index,
            COUNT(*) AS n_pos
        FROM read_parquet('{POS_PATH}')
        WHERE target = 1
        GROUP BY
            exposure_bucket_index,
            post_bucket_index,
            follow_10d_bucket_index
    )
    SELECT
        exposure_bucket_index,
        post_bucket_index,
        follow_10d_bucket_index,
        n_pos,
        ROUND(100.0 * n_pos / {total}, 4) AS pct_pos,
        ROUND({SAMPLE_SIZE}.0 * n_pos / {total}, 4) AS expected_n_in_sample,
        FLOOR({SAMPLE_SIZE}.0 * n_pos / {total}) AS floor_n_in_sample,
        CASE
            WHEN ({SAMPLE_SIZE}.0 * n_pos / {total}) < 1 THEN 1
            ELSE 0
        END AS expected_less_than_1
    FROM bucket_counts
    ORDER BY n_pos DESC, exposure_bucket_index
""").df()

print("\n=== BUCKET DISTRIBUTION, SORTED BY SIZE ===")
print(bucket_dist.to_string(index=False))

# SUMMARY

summary = con.execute(f"""
    WITH bucket_counts AS (
        SELECT
            exposure_bucket_index,
            COUNT(*) AS n_pos
        FROM read_parquet('{POS_PATH}')
        WHERE target = 1
        GROUP BY exposure_bucket_index
    ),
    bucket_stats AS (
        SELECT
            exposure_bucket_index,
            n_pos,
            {SAMPLE_SIZE}.0 * n_pos / {total} AS expected_n
        FROM bucket_counts
    )
    SELECT
        COUNT(*) AS n_positive_buckets,
        SUM(CASE WHEN expected_n < 1 THEN 1 ELSE 0 END) AS n_buckets_expected_less_than_1,
        SUM(CASE WHEN expected_n >= 1 THEN 1 ELSE 0 END) AS n_buckets_expected_at_least_1,
        MIN(n_pos) AS min_bucket_n_pos,
        MAX(n_pos) AS max_bucket_n_pos,
        ROUND(AVG(n_pos), 2) AS avg_bucket_n_pos
    FROM bucket_stats
""").df()

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

# DISTRIBUTION BY POST BUCKET ONLY

post_dist = con.execute(f"""
    SELECT
        post_bucket_index,
        COUNT(*) AS n_pos,
        ROUND(100.0 * COUNT(*) / {total}, 4) AS pct_pos,
        ROUND({SAMPLE_SIZE}.0 * COUNT(*) / {total}, 4) AS expected_n_in_sample
    FROM read_parquet('{POS_PATH}')
    WHERE target = 1
    GROUP BY post_bucket_index
    ORDER BY post_bucket_index
""").df()

print("\n=== DISTRIBUTION BY POST BUCKET ===")
print(post_dist.to_string(index=False))

# DISTRIBUTION BY FOLLOW BUCKET ONLY

follow_dist = con.execute(f"""
    SELECT
        follow_10d_bucket_index,
        COUNT(*) AS n_pos,
        ROUND(100.0 * COUNT(*) / {total}, 4) AS pct_pos,
        ROUND({SAMPLE_SIZE}.0 * COUNT(*) / {total}, 4) AS expected_n_in_sample
    FROM read_parquet('{POS_PATH}')
    WHERE target = 1
    GROUP BY follow_10d_bucket_index
    ORDER BY follow_10d_bucket_index
""").df()

print("\n=== DISTRIBUTION BY FOLLOW BUCKET ===")
print(follow_dist.to_string(index=False))


=== POSITIVE BUCKET SIZE DISTRIBUTION ===
Source: ..\..\..\datasets\balduf_anon_march_2026\positive_blocks_analysis_10d_mar2026_v3_anon.parquet
Target sample size: 1000

=== BUCKET DISTRIBUTION, SORTED BY SIZE ===
 exposure_bucket_index  post_bucket_index  follow_10d_bucket_index  n_pos  pct_pos  expected_n_in_sample  floor_n_in_sample  expected_less_than_1
                     0                  0                        0  28802  60.2288              602.2877              602.0                     0
                   100                  1                        0   2119   4.4311               44.3111               44.0                     0
                     1                  0                        1   1996   4.1739               41.7390               41.0                     0
                     3                  0                        3   1276   2.6683               26.6828               26.0                     0
                   200                  2              

## creazione pool dei positivi

In [ ]:
#creazione della pool dei positivi
#campiono, su 1000, tutti i bucket che hanno almeno 0.1% dei positivi (cioè almeno 1 bucket) e 
# prendo un numero di campioni proporzionale alla loro dimensione (floor(expected_n_in_sample))

# CONFIG

POS_PATH = Path("../../../datasets/balduf_anon_march_2026/positive_blocks_analysis_10d_mar2026_v3_anon.parquet")

OUT_DIR = Path("../../../datasets/balduf_anon_march_2026/rf_10000_pool")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_POS_POOL = OUT_DIR / "positive_10000_pool.parquet"
OUT_ALLOCATION = OUT_DIR / "positive_10000_pool_allocation.parquet"

SAMPLE_SIZE = 10000
MIN_PCT_TO_KEEP = 0.1
RANDOM_SEED = 42


# CREATE POSITIVE POOL

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TEMP TABLE pos AS
SELECT *
FROM read_parquet('{POS_PATH}')
WHERE target = 1
""")

# Distribuzione dei positivi per bucket

con.execute(f"""
CREATE OR REPLACE TEMP TABLE bucket_distribution AS
SELECT
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index,
    COUNT(*) AS n_pos,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_pos,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () * {SAMPLE_SIZE} AS expected_n_in_sample
FROM pos
GROUP BY
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index
""")

# Regola corretta:
# - se pct_pos >= 0.1%: prendi floor(expected)
# - se pct_pos < 0.1%: escludi bucket
con.execute(f"""
CREATE OR REPLACE TEMP TABLE allocation AS
SELECT
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index,
    n_pos,
    pct_pos,
    expected_n_in_sample,
    CASE
        WHEN pct_pos >= {MIN_PCT_TO_KEEP}
            THEN CAST(FLOOR(expected_n_in_sample) AS BIGINT)
        ELSE 0
    END AS n_to_sample
FROM bucket_distribution
""")

# Salva allocazione
con.execute(f"""
COPY (
    SELECT *
    FROM allocation
    ORDER BY n_to_sample DESC, n_pos DESC, exposure_bucket_index
)
TO '{OUT_ALLOCATION}'
(FORMAT PARQUET)
""")

# Campionamento casuale dentro ogni bucket
con.execute(f"""
CREATE OR REPLACE TEMP TABLE sampled_pos AS
WITH ranked AS (
    SELECT
        p.*,
        a.n_to_sample,
        ROW_NUMBER() OVER (
            PARTITION BY p.exposure_bucket_index
            ORDER BY random()
        ) AS rn
    FROM pos p
    INNER JOIN allocation a
        ON p.exposure_bucket_index = a.exposure_bucket_index
    WHERE a.n_to_sample > 0
)
SELECT
    *
EXCLUDE (n_to_sample, rn)
FROM ranked
WHERE rn <= n_to_sample
""")

# Salva pool positivo
con.execute(f"""
COPY sampled_pos
TO '{OUT_POS_POOL}'
(FORMAT PARQUET)
""")


# CHECKS

print("\n=== POSITIVE POOL CREATED ===")
print(f"Positive pool: {OUT_POS_POOL}")
print(f"Allocation table: {OUT_ALLOCATION}")

print("\n=== TOTAL SAMPLE ===")
print(con.execute("""
SELECT
    COUNT(*) AS sampled_positive_rows,
    COUNT(DISTINCT did_anon) AS sampled_distinct_dids,
    COUNT(DISTINCT exposure_bucket_index) AS sampled_buckets
FROM sampled_pos
""").df())

print("\n=== ALLOCATION SUMMARY ===")
print(con.execute("""
SELECT
    COUNT(*) AS total_positive_buckets,
    SUM(CASE WHEN n_to_sample > 0 THEN 1 ELSE 0 END) AS kept_buckets,
    SUM(CASE WHEN n_to_sample = 0 THEN 1 ELSE 0 END) AS excluded_buckets,
    SUM(n_to_sample) AS planned_sample_size
FROM allocation
""").df())



=== POSITIVE POOL CREATED ===
Positive pool: ..\..\..\datasets\balduf_anon_march_2026\rf_10000_pool\positive_10000_pool.parquet
Allocation table: ..\..\..\datasets\balduf_anon_march_2026\rf_10000_pool\positive_10000_pool_allocation.parquet

=== TOTAL SAMPLE ===
   sampled_positive_rows  sampled_distinct_dids  sampled_buckets
0                   9778                   9778               48

=== ALLOCATION SUMMARY ===
   total_positive_buckets  kept_buckets  excluded_buckets  planned_sample_size
0                     133          48.0              85.0               9778.0


## creazione pool negativi

In [ ]:
#creo anche quella dei negativi

import duckdb
from pathlib import Path

# CONFIG

NEG_PATH = Path("../../../datasets/balduf_anon_march_2026/negative_blocks_analysis_10d_mar2026_v3_anon.parquet")
OUT_DIR = Path("../../../datasets/balduf_anon_march_2026/rf_10000_pool")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOCATION_PATH = OUT_DIR / "positive_10000_pool_allocation.parquet"
OUT_NEG_POOL = OUT_DIR / "negative_10000_pool.parquet"

RANDOM_SEED = 42


# CREATE NEGATIVE POOL

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TEMP TABLE neg AS
SELECT *
FROM read_parquet('{NEG_PATH}')
WHERE target = 0
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE allocation AS
SELECT *
FROM read_parquet('{ALLOCATION_PATH}')
WHERE n_to_sample > 0
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE neg_availability AS
SELECT
    exposure_bucket_index,
    COUNT(*) AS n_neg_available
FROM neg
GROUP BY exposure_bucket_index
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE neg_sampling_check AS
SELECT
    a.exposure_bucket_index,
    a.post_bucket_index,
    a.follow_10d_bucket_index,
    a.n_pos,
    a.pct_pos,
    a.expected_n_in_sample,
    a.n_to_sample AS n_neg_to_sample,
    COALESCE(n.n_neg_available, 0) AS n_neg_available,
    CASE
        WHEN COALESCE(n.n_neg_available, 0) >= a.n_to_sample THEN 0
        ELSE 1
    END AS shortage_flag
FROM allocation a
LEFT JOIN neg_availability n
    ON a.exposure_bucket_index = n.exposure_bucket_index
""")

# Campiona negativi: stesso numero dei positivi per exposure_bucket_index
con.execute("""
CREATE OR REPLACE TEMP TABLE sampled_neg AS
WITH ranked AS (
    SELECT
        n.*,
        c.n_neg_to_sample,
        ROW_NUMBER() OVER (
            PARTITION BY n.exposure_bucket_index
            ORDER BY random()
        ) AS rn
    FROM neg n
    INNER JOIN neg_sampling_check c
        ON n.exposure_bucket_index = c.exposure_bucket_index
    WHERE c.n_neg_to_sample > 0
      AND c.shortage_flag = 0
)
SELECT
    *
EXCLUDE (n_neg_to_sample, rn)
FROM ranked
WHERE rn <= n_neg_to_sample
""")

con.execute(f"""
COPY sampled_neg
TO '{OUT_NEG_POOL}'
(FORMAT PARQUET)
""")


# =========================
# PRINT CHECKS ONLY
# =========================

print("\n=== NEGATIVE POOL CREATED ===")
print(f"Negative pool: {OUT_NEG_POOL}")

print("\n=== NEGATIVE SAMPLE TOTALS ===")
print(con.execute("""
SELECT
    COUNT(*) AS sampled_negative_rows,
    COUNT(DISTINCT did_anon) AS sampled_distinct_dids,
    COUNT(DISTINCT exposure_bucket_index) AS sampled_buckets
FROM sampled_neg
""").df())

print("\n=== SAMPLING CHECK SUMMARY ===")
print(con.execute("""
SELECT
    COUNT(*) AS buckets_requested,
    SUM(n_neg_to_sample) AS planned_negative_sample_size,
    SUM(shortage_flag) AS buckets_with_shortage,
    SUM(CASE WHEN shortage_flag = 0 THEN n_neg_to_sample ELSE 0 END) AS feasible_negative_sample_size
FROM neg_sampling_check
""").df())

print("\n=== SHORTAGE BUCKETS, IF ANY ===")
print(con.execute("""
SELECT
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index,
    n_neg_to_sample,
    n_neg_available,
    shortage_flag
FROM neg_sampling_check
WHERE shortage_flag = 1
ORDER BY n_neg_to_sample DESC, exposure_bucket_index
""").df())

print("\n=== FINAL NEGATIVE SAMPLE BY BUCKET ===")
print(con.execute("""
SELECT
    exposure_bucket_index,
    COUNT(*) AS sampled_negatives
FROM sampled_neg
GROUP BY exposure_bucket_index
ORDER BY sampled_negatives DESC, exposure_bucket_index
""").df())




=== NEGATIVE POOL CREATED ===
Negative pool: ..\..\..\datasets\balduf_anon_march_2026\rf_10000_pool\negative_10000_pool.parquet

=== NEGATIVE SAMPLE TOTALS ===
   sampled_negative_rows  sampled_distinct_dids  sampled_buckets
0                   9778                   9606               48

=== SAMPLING CHECK SUMMARY ===
   buckets_requested  planned_negative_sample_size  buckets_with_shortage  \
0                 48                        9778.0                    0.0   

   feasible_negative_sample_size  
0                         9778.0  

=== SHORTAGE BUCKETS, IF ANY ===
Empty DataFrame
Columns: [exposure_bucket_index, post_bucket_index, follow_10d_bucket_index, n_neg_to_sample, n_neg_available, shortage_flag]
Index: []

=== FINAL NEGATIVE SAMPLE BY BUCKET ===
    exposure_bucket_index  sampled_negatives
0                       0               6022
1                     100                443
2                       1                417
3                       3                266


# creazione pool 1000 no 00, 01

In [ ]:

# CREAZIONE DELLA POOL DEI POSITIVI (no 00, 01)
# - escludo i bucket 00 e 01
# - ricalcolo le percentuali solo sui bucket rimanenti
# - campiono proporzionalmente alla nuova distribuzione

# CONFIG

POS_PATH = Path(
    "../../../datasets/balduf_anon_march_2026/"
    "positive_blocks_analysis_10d_mar2026_v3_anon.parquet"
)

OUT_DIR = Path("../../../datasets/balduf_anon_march_2026/rf_10000_no00_01_pool")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_POS_POOL = OUT_DIR / "positive_10000_pool.parquet"
OUT_ALLOCATION = OUT_DIR / "positive_10000_pool_allocation.parquet"

SAMPLE_SIZE = 10000
MIN_PCT_TO_KEEP = 0.1
RANDOM_SEED = 42

# exposure_bucket_index = post_bucket_index * 100 + follow_10d_bucket_index
# 00 = exposure_bucket_index 0
# 01 = exposure_bucket_index 1
EXCLUDED_BUCKETS = (0, 1)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TEMP TABLE pos_all AS
SELECT *
FROM read_parquet('{POS_PATH}')
WHERE target = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE pos AS
SELECT *
FROM pos_all
WHERE exposure_bucket_index NOT IN {EXCLUDED_BUCKETS}
""")


# 2. Distribuzione dei positivi per bucket dopo l'esclusione

con.execute(f"""
CREATE OR REPLACE TEMP TABLE bucket_distribution AS
SELECT
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index,
    COUNT(*) AS n_pos,

    COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER () AS pct_pos,

    COUNT(*) * 1.0
        / SUM(COUNT(*)) OVER ()
        * {SAMPLE_SIZE} AS expected_n_in_sample

FROM pos
GROUP BY
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index
""")


# 3. Allocazione del campione
#    - se pct_pos >= 0.1%: prendo floor(expected)
#    - se pct_pos < 0.1%: escludo il bucket
#    - LEAST impedisce di chiedere più righe di quelle disponibili

con.execute(f"""
CREATE OR REPLACE TEMP TABLE allocation AS
SELECT
    exposure_bucket_index,
    post_bucket_index,
    follow_10d_bucket_index,
    n_pos,
    pct_pos,
    expected_n_in_sample,

    CASE
        WHEN pct_pos >= {MIN_PCT_TO_KEEP}
            THEN LEAST(
                n_pos,
                CAST(FLOOR(expected_n_in_sample) AS BIGINT)
            )
        ELSE 0
    END AS n_to_sample

FROM bucket_distribution
""")


# 4. Salvo la tabella di allocazione

con.execute(f"""
COPY (
    SELECT *
    FROM allocation
    ORDER BY
        n_to_sample DESC,
        n_pos DESC,
        exposure_bucket_index
)
TO '{OUT_ALLOCATION}'
(FORMAT PARQUET)
""")


# 5. Campionamento dentro ogni bucket
#
# Uso hash(did_anon, RANDOM_SEED) invece di random():
# il campionamento rimane pseudocasuale ma è riproducibile.

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sampled_pos AS
WITH ranked AS (
    SELECT
        p.*,
        a.n_to_sample,

        ROW_NUMBER() OVER (
            PARTITION BY p.exposure_bucket_index
            ORDER BY hash(p.did_anon, {RANDOM_SEED})
        ) AS rn

    FROM pos p
    INNER JOIN allocation a
        ON p.exposure_bucket_index = a.exposure_bucket_index

    WHERE a.n_to_sample > 0
)

SELECT
    *
    EXCLUDE (n_to_sample, rn)

FROM ranked
WHERE rn <= n_to_sample
""")


# 6. Salvo la nuova pool dei positivi

con.execute(f"""
COPY sampled_pos
TO '{OUT_POS_POOL}'
(FORMAT PARQUET)
""")

# CHECKS

print("\n=== BEFORE / AFTER EXCLUSION ===")
print(con.execute("""
SELECT
    (SELECT COUNT(*) FROM pos_all) AS original_positive_rows,
    (SELECT COUNT(*) FROM pos) AS eligible_positive_rows,
    (SELECT COUNT(*) FROM pos_all) - (SELECT COUNT(*) FROM pos) AS removed_positive_rows,
    (SELECT COUNT(DISTINCT exposure_bucket_index) FROM pos_all) AS original_buckets,
    (SELECT COUNT(DISTINCT exposure_bucket_index) FROM pos) AS eligible_buckets
""").df())


print("\n=== TOTAL SAMPLE ===")
print(con.execute("""
SELECT
    COUNT(*) AS sampled_positive_rows,
    COUNT(DISTINCT did_anon) AS sampled_distinct_dids,
    COUNT(DISTINCT exposure_bucket_index) AS sampled_buckets
FROM sampled_pos
""").df())


print("\n=== ALLOCATION SUMMARY ===")
print(con.execute("""
SELECT
    COUNT(*) AS total_eligible_positive_buckets,
    SUM(CASE WHEN n_to_sample > 0 THEN 1 ELSE 0 END) AS kept_buckets,
    SUM(CASE WHEN n_to_sample = 0 THEN 1 ELSE 0 END) AS excluded_small_buckets,
    SUM(n_to_sample) AS planned_sample_size
FROM allocation
""").df())


print("\n=== SAMPLE DISTRIBUTION BY BUCKET ===")
print(con.execute("""
SELECT
    a.exposure_bucket_index,
    a.post_bucket_index,
    a.follow_10d_bucket_index,
    a.n_pos,
    ROUND(a.pct_pos, 4) AS pct_pos_after_exclusion,
    ROUND(a.expected_n_in_sample, 2) AS expected_n_in_sample,
    a.n_to_sample,
    COUNT(s.did_anon) AS sampled_rows
FROM allocation a
LEFT JOIN sampled_pos s
    ON a.exposure_bucket_index = s.exposure_bucket_index
GROUP BY
    a.exposure_bucket_index,
    a.post_bucket_index,
    a.follow_10d_bucket_index,
    a.n_pos,
    a.pct_pos,
    a.expected_n_in_sample,
    a.n_to_sample
ORDER BY
    a.n_to_sample DESC,
    a.n_pos DESC,
    a.exposure_bucket_index
""").df())


print("\n=== FINAL CHECK: REMOVED BUCKETS MUST NOT APPEAR IN SAMPLE ===")
print(con.execute(f"""
SELECT COUNT(*) AS forbidden_rows_in_sample
FROM sampled_pos
WHERE exposure_bucket_index IN {EXCLUDED_BUCKETS}
""").df())


con.close()

NameError: name 'Path' is not defined

In [ ]:
# CREAZIONE DELLA POOL DEI NEGATIVI CON DID DISTINTI
# - esclude i bucket 00 e 01
# - mantiene una sola osservazione per ogni did_anon
# - campiona lo stesso numero di negativi dei positivi
#   per ciascun exposure_bucket_index

# CONFIG

NEG_PATH = Path(
    "../../../datasets/balduf_anon_march_2026/"
    "negative_blocks_analysis_10d_mar2026_v3_anon.parquet"
)

OUT_DIR = Path("../../../datasets/balduf_anon_march_2026/rf_10000_no00_01_pool")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Allocation positiva già rigenerata senza bucket 00 e 01
ALLOCATION_PATH = OUT_DIR / "positive_10000_pool_allocation.parquet"

OUT_NEG_POOL = OUT_DIR / "negative_10000_pool.parquet"
OUT_NEG_CHECK = OUT_DIR / "negative_10000_pool_sampling_check.parquet"

RANDOM_SEED = 42

# 00 = post_bucket_index 0, follow_10d_bucket_index 0
# 01 = post_bucket_index 0, follow_10d_bucket_index 1
EXCLUDED_BUCKETS = (0, 1)

# CREATE NEGATIVE POOL

con = duckdb.connect()

# 1. Leggo tutti i negativi

con.execute(f"""
CREATE OR REPLACE TEMP TABLE neg_all AS
SELECT *
FROM read_parquet('{NEG_PATH}')
WHERE target = 0
""")

# 2. Escludo i bucket 00 e 01

con.execute(f"""
CREATE OR REPLACE TEMP TABLE neg_eligible AS
SELECT *
FROM neg_all
WHERE exposure_bucket_index NOT IN {EXCLUDED_BUCKETS}
""")

# 3. Tengo una sola osservazione per ogni DID negativo
#
# Ogni DID negativo può comparire in più giorni indice.
# Qui scelgo una sola riga in modo pseudocasuale ma
# riproducibile tramite hash.
#
# Dopo questo passaggio:
# - ogni did_anon compare una sola volta;
# - ciascun DID appartiene a un solo exposure bucket;
# - il campionamento successivo non può duplicare account.

con.execute(f"""
CREATE OR REPLACE TEMP TABLE neg_unique AS
SELECT *
FROM neg_eligible
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY did_anon
    ORDER BY
        hash(did_anon, event_time, exposure_bucket_index, {RANDOM_SEED}),
        event_time,
        exposure_bucket_index
) = 1
""")


# 4. Leggo l'allocation positiva.
#    La numerosità richiesta per ciascun bucket deriva
#    direttamente dalla pool positiva.

con.execute(f"""
CREATE OR REPLACE TEMP TABLE allocation AS
SELECT *
FROM read_parquet('{ALLOCATION_PATH}')
WHERE n_to_sample > 0
  AND exposure_bucket_index NOT IN {EXCLUDED_BUCKETS}
""")


# 5. Calcolo quanti DID negativi distinti sono disponibili
#    in ciascun bucket dopo la deduplicazione.

con.execute("""
CREATE OR REPLACE TEMP TABLE neg_availability AS
SELECT
    exposure_bucket_index,
    COUNT(*) AS n_neg_available,
    COUNT(DISTINCT did_anon) AS n_distinct_neg_available
FROM neg_unique
GROUP BY exposure_bucket_index
""")


# 6. Controllo se ogni bucket contiene abbastanza negativi
#    distinti per replicare l'allocation positiva.

con.execute("""
CREATE OR REPLACE TEMP TABLE neg_sampling_check AS
SELECT
    a.exposure_bucket_index,
    a.post_bucket_index,
    a.follow_10d_bucket_index,
    a.n_pos,
    a.pct_pos,
    a.expected_n_in_sample,

    a.n_to_sample AS n_pos_sampled,
    a.n_to_sample AS n_neg_to_sample,

    COALESCE(n.n_distinct_neg_available, 0) AS n_distinct_neg_available,

    CASE
        WHEN COALESCE(n.n_distinct_neg_available, 0) >= a.n_to_sample
            THEN 0
        ELSE 1
    END AS shortage_flag

FROM allocation a
LEFT JOIN neg_availability n
    ON a.exposure_bucket_index = n.exposure_bucket_index
""")


# 7. Salvo il controllo di disponibilità

con.execute(f"""
COPY (
    SELECT *
    FROM neg_sampling_check
    ORDER BY
        shortage_flag DESC,
        n_neg_to_sample DESC,
        exposure_bucket_index
)
TO '{OUT_NEG_CHECK}'
(FORMAT PARQUET)
""")


# 8. Interrompo lo script se, dopo aver imposto DID distinti,
#    qualche bucket non contiene abbastanza negativi.
#
#    È preferibile fermarsi invece di creare silenziosamente
#    una pool non bilanciata rispetto ai positivi.

shortage_summary = con.execute("""
SELECT
    COUNT(*) AS buckets_with_shortage,
    SUM(
        CASE
            WHEN shortage_flag = 1
                THEN n_neg_to_sample - n_distinct_neg_available
            ELSE 0
        END
    ) AS missing_negative_dids
FROM neg_sampling_check
WHERE shortage_flag = 1
""").df()

n_shortage_buckets = int(shortage_summary["buckets_with_shortage"].iloc[0])

if n_shortage_buckets > 0:
    print("\n=== ERROR: SHORTAGE OF DISTINCT NEGATIVE DIDS ===")
    print(shortage_summary)

    print("\n=== SHORTAGE BUCKETS ===")
    print(con.execute("""
    SELECT
        exposure_bucket_index,
        post_bucket_index,
        follow_10d_bucket_index,
        n_neg_to_sample,
        n_distinct_neg_available,
        n_neg_to_sample - n_distinct_neg_available AS missing_dids
    FROM neg_sampling_check
    WHERE shortage_flag = 1
    ORDER BY missing_dids DESC, exposure_bucket_index
    """).df())

    con.close()

    raise RuntimeError(
        "Impossibile creare una pool negativa bilanciata con DID distinti: "
        "almeno un exposure bucket non contiene abbastanza account negativi unici."
    )


# 9. Campiono i negativi distinti:
#    stesso numero dei positivi in ogni exposure bucket.

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sampled_neg AS
WITH ranked AS (
    SELECT
        n.*,
        c.n_neg_to_sample,

        ROW_NUMBER() OVER (
            PARTITION BY n.exposure_bucket_index
            ORDER BY
                hash(n.did_anon, n.event_time, {RANDOM_SEED}),
                n.did_anon
        ) AS rn

    FROM neg_unique n
    INNER JOIN neg_sampling_check c
        ON n.exposure_bucket_index = c.exposure_bucket_index

    WHERE c.n_neg_to_sample > 0
)

SELECT
    *
    EXCLUDE (n_neg_to_sample, rn)

FROM ranked
WHERE rn <= n_neg_to_sample
""")


# 10. Salvo la pool negativa finale

con.execute(f"""
COPY sampled_neg
TO '{OUT_NEG_POOL}'
(FORMAT PARQUET)
""")


# CHECKS

print("\n=== NEGATIVE POOL CREATED ===")
print(f"Negative pool: {OUT_NEG_POOL}")
print(f"Sampling check: {OUT_NEG_CHECK}")


print("\n=== ORIGINAL NEGATIVE DATASET ===")
print(con.execute("""
SELECT
    COUNT(*) AS original_negative_rows,
    COUNT(DISTINCT did_anon) AS original_distinct_dids,
    COUNT(DISTINCT exposure_bucket_index) AS original_buckets
FROM neg_all
""").df())

print("\n=== FINAL NEGATIVE SAMPLE TOTALS ===")
print(con.execute("""
SELECT
    COUNT(*) AS sampled_negative_rows,
    COUNT(DISTINCT did_anon) AS sampled_distinct_dids,
    COUNT(*) - COUNT(DISTINCT did_anon) AS duplicated_dids,
    COUNT(DISTINCT exposure_bucket_index) AS sampled_buckets
FROM sampled_neg
""").df())


print("\n=== FINAL CHECK: NO DUPLICATED DID IN NEGATIVE POOL ===")
print(con.execute("""
SELECT
    did_anon,
    COUNT(*) AS n_occurrences
FROM sampled_neg
GROUP BY did_anon
HAVING COUNT(*) > 1
ORDER BY n_occurrences DESC, did_anon
""").df())


print("\n=== FINAL CHECK: REMOVED BUCKETS MUST NOT APPEAR ===")
print(con.execute(f"""
SELECT
    COUNT(*) AS forbidden_rows_in_sample
FROM sampled_neg
WHERE exposure_bucket_index IN {EXCLUDED_BUCKETS}
""").df())


con.close()


=== NEGATIVE POOL CREATED ===
Negative pool: ..\..\..\datasets\balduf_anon_march_2026\rf_10000_no00_01_pool\negative_10000_pool.parquet
Sampling check: ..\..\..\datasets\balduf_anon_march_2026\rf_10000_no00_01_pool\negative_10000_pool_sampling_check.parquet

=== ORIGINAL NEGATIVE DATASET ===
   original_negative_rows  original_distinct_dids  original_buckets
0                 4031665                  366515               144

=== FINAL NEGATIVE SAMPLE TOTALS ===
   sampled_negative_rows  sampled_distinct_dids  duplicated_dids  \
0                   9728                   9728                0   

   sampled_buckets  
0               67  

=== FINAL CHECK: NO DUPLICATED DID IN NEGATIVE POOL ===
Empty DataFrame
Columns: [did_anon, n_occurrences]
Index: []

=== FINAL CHECK: REMOVED BUCKETS MUST NOT APPEAR ===
   forbidden_rows_in_sample
0                         0


# creazione pool no 00 bilanciata con il max numero possibile restando 1:1

In [3]:
from pathlib import Path
import duckdb

RANDOM_SEED = 42

DATA_DIR = Path("../../../datasets/balduf_anon_march_2026")
POS_PATH = DATA_DIR / "positive_blocks_analysis_10d_mar2026_v3_anon.parquet"
NEG_PATH = DATA_DIR / "negative_blocks_analysis_10d_mar2026_v3_anon.parquet"

OUT_DIR = DATA_DIR / "rf_max_pool_no_00"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_POS_POOL = OUT_DIR / "positive_max_pool_no_00.parquet"
OUT_NEG_POOL = OUT_DIR / "negative_max_pool_no_00.parquet"
OUT_ALLOCATION = OUT_DIR / "max_pool_no_00_allocation.parquet"

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TEMP TABLE pos_unique AS
WITH base AS (
    SELECT
        *,
        CASE
            WHEN post_bucket_index = 0 AND follow_10d_bucket_index > 0 THEN '0P'
            WHEN post_bucket_index > 0 AND follow_10d_bucket_index = 0 THEN 'P0'
            WHEN post_bucket_index > 0 AND follow_10d_bucket_index > 0 THEN 'PP'
        END AS exposure_corner
    FROM read_parquet('{POS_PATH.as_posix()}')
    WHERE target = 1
      AND NOT (post_bucket_index = 0 AND follow_10d_bucket_index = 0)
)
SELECT *
FROM base
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY did_anon
    ORDER BY hash(did_anon, event_time, post_bucket_index, follow_10d_bucket_index, {RANDOM_SEED})
) = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE neg_unique AS
WITH base AS (
    SELECT
        *,
        CASE
            WHEN post_bucket_index = 0 AND follow_10d_bucket_index > 0 THEN '0P'
            WHEN post_bucket_index > 0 AND follow_10d_bucket_index = 0 THEN 'P0'
            WHEN post_bucket_index > 0 AND follow_10d_bucket_index > 0 THEN 'PP'
        END AS exposure_corner
    FROM read_parquet('{NEG_PATH.as_posix()}')
    WHERE target = 0
      AND NOT (post_bucket_index = 0 AND follow_10d_bucket_index = 0)
)
SELECT *
FROM base
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY did_anon
    ORDER BY hash(did_anon, event_time, post_bucket_index, follow_10d_bucket_index, {RANDOM_SEED})
) = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE pos_counts AS
SELECT exposure_corner, COUNT(*) AS n_pos
FROM pos_unique
GROUP BY exposure_corner
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE neg_counts AS
SELECT exposure_corner, COUNT(*) AS n_neg
FROM neg_unique
GROUP BY exposure_corner
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE allocation AS
WITH limits AS (
    SELECT LEAST(
        (SELECT MIN(n_pos) FROM pos_counts WHERE exposure_corner IN ('0P', 'P0', 'PP')),
        (SELECT MIN(n_neg) FROM neg_counts WHERE exposure_corner IN ('0P', 'P0', 'PP'))
    ) AS n_to_sample
)
SELECT
    p.exposure_corner,
    p.n_pos,
    n.n_neg,
    l.n_to_sample
FROM pos_counts p
JOIN neg_counts n USING (exposure_corner)
CROSS JOIN limits l
WHERE p.exposure_corner IN ('0P', 'P0', 'PP')
ORDER BY p.exposure_corner
""")

con.execute(f"""
COPY allocation
TO '{OUT_ALLOCATION.as_posix()}'
(FORMAT PARQUET)
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sampled_pos AS
WITH ranked AS (
    SELECT
        p.*,
        a.n_to_sample,
        ROW_NUMBER() OVER (
            PARTITION BY p.exposure_corner
            ORDER BY hash(p.did_anon, p.event_time, {RANDOM_SEED})
        ) AS rn
    FROM pos_unique p
    JOIN allocation a USING (exposure_corner)
)
SELECT * EXCLUDE (n_to_sample, rn)
FROM ranked
WHERE rn <= n_to_sample
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sampled_neg AS
WITH ranked AS (
    SELECT
        n.*,
        a.n_to_sample,
        ROW_NUMBER() OVER (
            PARTITION BY n.exposure_corner
            ORDER BY hash(n.did_anon, n.event_time, {RANDOM_SEED})
        ) AS rn
    FROM neg_unique n
    JOIN allocation a USING (exposure_corner)
)
SELECT * EXCLUDE (n_to_sample, rn)
FROM ranked
WHERE rn <= n_to_sample
""")

con.execute(f"""
COPY sampled_pos
TO '{OUT_POS_POOL.as_posix()}'
(FORMAT PARQUET)
""")

con.execute(f"""
COPY sampled_neg
TO '{OUT_NEG_POOL.as_posix()}'
(FORMAT PARQUET)
""")

print("\n=== ALLOCATION ===")
print(con.execute("""
SELECT *
FROM allocation
ORDER BY exposure_corner
""").df())

print("\n=== POSITIVE POOL ===")
print(con.execute("""
SELECT
    COUNT(*) AS rows_,
    COUNT(DISTINCT did_anon) AS distinct_dids
FROM sampled_pos
""").df())

print(con.execute("""
SELECT
    exposure_corner,
    COUNT(*) AS n
FROM sampled_pos
GROUP BY exposure_corner
ORDER BY exposure_corner
""").df())

print("\n=== NEGATIVE POOL ===")
print(con.execute("""
SELECT
    COUNT(*) AS rows_,
    COUNT(DISTINCT did_anon) AS distinct_dids
FROM sampled_neg
""").df())

print(con.execute("""
SELECT
    exposure_corner,
    COUNT(*) AS n
FROM sampled_neg
GROUP BY exposure_corner
ORDER BY exposure_corner
""").df())

con.close()



=== ALLOCATION ===
  exposure_corner  n_pos   n_neg  n_to_sample
0              0P   5914   51158         4775
1              P0   4775   82803         4775
2              PP   8330  148169         4775

=== POSITIVE POOL ===
   rows_  distinct_dids
0  14325          14325
  exposure_corner     n
0              0P  4775
1              P0  4775
2              PP  4775

=== NEGATIVE POOL ===
   rows_  distinct_dids
0  14325          14325
  exposure_corner     n
0              0P  4775
1              P0  4775
2              PP  4775
